<a href="https://colab.research.google.com/github/abhinav7056/Celebal-Technolgies-Internship/blob/main/week2_Abhinav_Garg.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from statsmodels.tsa.holtwinters import ExponentialSmoothing
import warnings
import os
import kagglehub
warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.labelsize"] = 11
print("Libraries imported successfully!")

In [ ]:
import kagglehub

dataset_path = kagglehub.dataset_download(
    "nalisha/tesla-ea-deliveries-and-production-data20152025"
)

print("Dataset downloaded successfully!")
print("File location:", dataset_path)

In [ ]:
import os

folder_path = '/kaggle/input/tesla-ea-deliveries-and-production-data20152025'

files = os.listdir(folder_path)

print("Files in dataset folder:")
print(files)

In [ ]:
file_path = '/kaggle/input/tesla-ea-deliveries-and-production-data20152025/tesla_deliveries_dataset_2015_2025.csv'

df = pd.read_csv(file_path)

df.columns = df.columns.str.strip()

print("Dataset loaded successfully!")
print("Number of rows and columns:", df.shape)

### **Data Preprocessing**


Data Info

In [ ]:
df.info()

FInding Missing Values

In [ ]:
print(df.isnull().mean())

Remove Duplications

In [ ]:
initial_rows = len(df)
df.drop_duplicates(inplace=True)
print(f"Removed {initial_rows - len(df)} duplicate rows.")

In [ ]:
df.dropna()

In [ ]:
print(df.dtypes)

### **Exploratory Data Analysis (EDA)**

In [ ]:
plt.figure(figsize=(10, 4))
sns.histplot(df['Avg_Price_USD'], kde=True, color='skyblue')
plt.title("Distribution of Tesla's Average Price", fontweight='bold')
plt.xlabel("Average Price (USD)")
plt.ylabel("Frequency")
plt.show()

In [ ]:
production_data = df.groupby(['Year', 'Model'])['Production_Units'].sum().reset_index()

plt.figure(figsize=(10, 6))

sns.barplot(
    data=production_data,
    x='Year',
    y='Production_Units',
    hue='Model',
    palette='viridis'
)

plt.title('Tesla Production by Model Over the Years')
plt.xlabel('Year')
plt.ylabel('Production Units')

plt.grid(axis='y', linestyle='--', alpha=0.6)

plt.legend(title='Model Name', loc='upper left', bbox_to_anchor=(1, 1))

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

num_columns = df.select_dtypes(include=[np.number]).columns

sns.heatmap(
    df[num_columns].corr(),
    annot=True,
    cmap='Blues',
    fmt='.2f',
    linewidths=0.5
)

plt.title('Heatmap of Feature Correlation')

plt.show()

### **Time Series Aggregation & Clean Timeline Construction**

In [ ]:
df['Date'] = pd.to_datetime(
    df['Year'].astype(str) + '-' +
    df['Month'].astype(str) + '-01'
)

monthly_data = df.groupby('Date').agg({
    'Estimated_Deliveries': 'sum',
    'Production_Units': 'sum',
    'Avg_Price_USD': 'mean',
    'Battery_Capacity_kWh': 'mean',
    'Range_km': 'mean'
})

monthly_data = monthly_data.sort_index()

print("Monthly time series data created successfully!")
print("Dataset shape:", monthly_data.shape)

print(monthly_data.head(4))

### **Feature Engineering**

In [ ]:
feature_data = monthly_data.copy()

feature_data['Delivery_Change'] = (
    feature_data['Estimated_Deliveries'].diff()
)

feature_data['Month'] = feature_data.index.month

feature_data['Previous_Delivery'] = (
    feature_data['Estimated_Deliveries'].shift(1)
)

feature_data['Production_Previous'] = (
    feature_data['Production_Units'].shift(1)
)

feature_data['Price_Previous'] = (
    feature_data['Avg_Price_USD'].shift(1)
)

feature_data['Change_Lag_1'] = (
    feature_data['Delivery_Change'].shift(1)
)

feature_data['Change_Lag_2'] = (
    feature_data['Delivery_Change'].shift(2)
)

feature_data['Change_Lag_12'] = (
    feature_data['Delivery_Change'].shift(12)
)

feature_data.drop(
    columns=['Production_Units', 'Avg_Price_USD'],
    inplace=True,
    errors='ignore'
)

feature_data.dropna(inplace=True)

print("Feature engineering completed successfully!")
print(
    feature_data[
        ['Estimated_Deliveries',
         'Delivery_Change',
         'Previous_Delivery']
    ].head(3)
)

### **Chronological Train-Test Horizon Splitting**

In [ ]:
X = feature_data.drop(columns=['Estimated_Deliveries'])
y = feature_data['Estimated_Deliveries']

test_size = 12

X_train = X.iloc[:-test_size]
X_test = X.iloc[-test_size:]

y_train = y.iloc[:-test_size]
y_test = y.iloc[-test_size:]

print("Data split completed successfully!\n")

print(
    "Training Data :",
    X_train.index.min().strftime('%Y-%m'),
    "to",
    X_train.index.max().strftime('%Y-%m')
)

print(
    "Testing Data :",
    X_test.index.min().strftime('%Y-%m'),
    "to",
    X_test.index.max().strftime('%Y-%m')
)

print("\nTraining samples:", len(X_train))
print("Testing samples:", len(X_test))

### **Machine Learning - Hyperparameter Tuning**

In [ ]:
gb_model = GradientBoostingRegressor(random_state=42)

params = {
    'n_estimators': [50, 100, 150],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 4, 5],
    'min_samples_split': [2, 5]
}

time_split = TimeSeriesSplit(n_splits=3)

model_search = RandomizedSearchCV(
    estimator=gb_model,
    param_distributions=params,
    n_iter=10,
    cv=time_split,
    scoring='neg_mean_squared_error',
    random_state=42,
    n_jobs=-1
)

print("Finding best parameters for Gradient Boosting Model...")

model_search.fit(X_train, y_train)

best_model = model_search.best_estimator_

print("Best Parameters Found:")
print(model_search.best_params_)

predictions = best_model.predict(X_test)

### **Statistical - Holt-Winters Modeling**

In [ ]:
time_model = ExponentialSmoothing(
    y_train,
    seasonal_periods=12,
    trend='add',
    seasonal='add'
)

fitted_model = time_model.fit()

time_predictions = fitted_model.forecast(steps=test_size)

print("Exponential Smoothing model trained successfully!")

### **Performance & Presentation Evaluation Dashboard**

In [ ]:
def model_metrics(actual, predicted, model_name):
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mae = mean_absolute_error(actual, predicted)
    r2 = r2_score(actual, predicted)

    return {
        'Model': model_name,
        'RMSE': round(rmse, 2),
        'MAE': round(mae, 2),
        'R2 Score': round(r2, 2)
    }


results = pd.DataFrame([
    model_metrics(
        y_test,
        predictions,
        'Gradient Boosting Model'
    ),

    model_metrics(
        y_test,
        time_predictions,
        'Exponential Smoothing'
    )
])

print("=" * 20, "MODEL PERFORMANCE", "=" * 20)
print(results.to_string(index=False))
print("=" * 60)


plt.figure(figsize=(12, 6))

plt.plot(
    y_test.index,
    y_test.values,
    label='Actual Deliveries',
    marker='o'
)

plt.plot(
    y_test.index,
    predictions,
    label='ML Prediction',
    marker='s'
)

plt.plot(
    y_test.index,
    time_predictions.values,
    label='Time Series Prediction',
    marker='^'
)

plt.title('Tesla Deliveries Prediction Comparison')
plt.xlabel('Date')
plt.ylabel('Estimated Deliveries')

plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)

plt.xticks(
    y_test.index,
    [date.strftime('%Y-%m') for date in y_test.index],
    rotation=30
)

plt.tight_layout()
plt.show()